# 01 · Synthetic data exploration

The records in `data/synthetic/` are generated **by the model itself** (`scripts/generate_synthetic_data.py`) from a documented synthetic truth. Exploring them checks that the generator behaves sensibly; it says nothing about real offshore facilities.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
from offshore_risk import load_config
cfg = load_config()
from offshore_risk.data import synthetic
d = synthetic.load()
log, tests = d['incident_log'], d['proof_tests']
log.head()

,event_id,date,year,platform,category,subcategory,outcome,downtime_h,lost_production_bbl,direct_cost_usd,business_interruption_usd,liquid_released_bbl
0,SYN-0001,2014-03-07,2014,UCP,mechanical,spurious_trip,spurious_trip,15.0,24716.0,150000.0,1433800.0,0.0
1,SYN-0002,2014-03-26,2014,UCP,mechanical,compressor,compressor_train_failure,87.5,55688.0,1088100.0,3230500.0,0.0
2,SYN-0003,2014-05-15,2014,PPB,mechanical,spurious_trip,spurious_trip,5.2,3877.0,150000.0,224900.0,0.0
3,SYN-0004,2014-06-30,2014,PPB,mechanical,spurious_trip,spurious_trip,9.7,7145.0,150000.0,414500.0,0.0
4,SYN-0005,2014-09-27,2014,UCP,mechanical,spurious_trip,spurious_trip,2.8,4610.0,150000.0,267400.0,0.0


In [2]:
log.groupby(['year', 'subcategory']).size().unstack(fill_value=0)

subcategory,compressor,power_loss,release,spurious_trip,weather
year,,,,,
2014,3,0,0,5,0
2015,2,0,0,1,0
2016,1,0,2,4,0
2017,0,0,1,11,0
2018,2,1,0,4,0
2019,2,0,2,8,0
2020,1,1,2,9,0
2021,1,0,0,2,0
2022,1,0,1,6,1


In [3]:
summary = log.groupby('subcategory').agg(events=('event_id', 'size'), median_downtime_h=('downtime_h', 'median'),
    total_lost_bbl=('lost_production_bbl', 'sum'), direct_cost_usd=('direct_cost_usd', 'sum'))
summary.sort_values('events', ascending=False)

,events,median_downtime_h,total_lost_bbl,direct_cost_usd
subcategory,,,,
spurious_trip,58,9.75,785348.0,8700000.0
compressor,14,214.00,2220614.0,15367400.0
release,11,52.80,951294.0,3332300.0
weather,5,68.30,587831.0,5000000.0
power_loss,2,31.75,104550.0,1955700.0


In [4]:
log['outcome'][log.subcategory == 'release'].value_counts()

outcome
controlled_release    10
minor_fire             1
Name: count, dtype: int64

In [5]:
t = tests.assign(fail=tests.result != 'pass').groupby('equipment_type')['fail'].agg(['sum', 'size', 'mean'])
t.columns = ['failures', 'tests', 'observed_rate']; t

,failures,tests,observed_rate
equipment_type,,,
diesel_firewater_pump,8,626,0.012780
electric_firewater_pump,36,1252,0.028754


**Observation.** Twelve years contain about a dozen releases, at most a minor fire and no major fire or explosion. A frequency estimated from this record alone would say 'major fires never happen'; this is why generic priors and event-tree logic, not only the operator's own history, are needed for major-accident risk. By contrast, spurious trips and pump start tests are numerous enough for the records to dominate the prior (see notebook 03).